<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 5 — Rhythms and Resampling</div>
</div>

#### Spectra, from events to a regular series, testing a relationship with surrogates, and bootstrap intervals for a difference.
***

## Where this fits

Lecture 5 asked whether successive values are independent; Lecture 6 turned a time series into a spectrum and followed it in time. Today you practise both on signals where **you know the answer**, because you made them, and meet the tools Lecture 8 applies to heartbeats:

0. are successive values independent? a shuffle test, and what it means for an uncertainty;
1. which frequencies, and when? spectra, the short-time Fourier transform, and averaged spectra;
2. from events to a regular series: turning a list of "beat" times into a signal you can take a spectrum of;
3. is a relationship real? cross-correlation and a surrogate test;
4. how big is a difference between two groups? a bootstrap interval.

These are the tools you'll need for Exercise 2, which is released after this lab. **Nothing here is assessed.**

## How to use this notebook today

**Core (about 100 minutes):** Section 0, Section 1(a)–(d), Section 2(a)–(d) and Section 3(a)–(b). The core cells already contain most of the code: fill in each `____`, run it, and compare with the **Check**. Everything marked *Optional* (1(e)–(f), 2(e), 3(c), Section 4 and the tide at the end) can wait, or be skipped. Section 4 is worth doing before you start Exercise 2.

**New to some of the Python?** Start with `week5_python_toolkit.ipynb` (15 minutes): `rfft`, `spectrogram` and `welch`, interpolating uneven times, `np.roll`, and timing code. If the basics themselves (arrays, slicing, masks, loops, functions) are shaky, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises them one at a time; each section below names the ones it uses.

Each task has a **Check** with the number you should get, and some have a collapsed hint.

In [ ]:
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week5/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, stats
from scipy.interpolate import CubicSpline

FS = 4.0                                   # samples per second for every regular series today

## <span style="color:#B5121B">0. Are successive values independent? (15 minutes)</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 1 (NumPy arrays); Section 5 (loops that fill an array); Section 6 (functions). Each section takes 10–15 minutes.

Lecture 5 (slide 22) tested whether successive values are independent by **shuffling** them: a shuffle keeps every value (the same mean, spread and histogram) but destroys any order. If the order matters, the real series looks different from its shuffled copies.

Below is a record in which each value is 0.8 times the previous one plus fresh noise, so neighbours are strongly related, and the true mean is 0.

**(a)** Compute the lag-1 autocorrelation $r_1$ (the correlation of each value with the next). Shuffle the record 2000 times and compute $r_1$ for each shuffled copy. How often does a shuffle reach the real value?

**(b)** So the values are not independent. Does that matter? Compute the naive standard error of the mean, $\sigma/\sqrt{N}$, then the standard error of the means of blocks of 50 values. Which do you believe? (Exercise 2, Task 1.2.)

In [ ]:
rng0 = np.random.default_rng(11)
a, n0 = 0.8, 2000
x0 = np.empty(n0); x0[0] = rng0.standard_normal()
for i in range(1, n0):
    x0[i] = a * x0[i - 1] + rng0.standard_normal()

In [ ]:
def lag1(x):
    """Lag-1 autocorrelation: the correlation of each value with the next."""
    d = x - x.mean()
    return np.sum(d[:-1] * d[1:]) / np.sum(d**2)

rng_s = np.random.default_rng(12)
r_real = lag1(x0)
r_shuf = np.array([lag1(rng_s.permutation(____)) for _ in range(2000)])     # 2000 shuffled copies
print(f"r1 = {r_real:.2f}; {np.sum(np.abs(r_shuf) >= abs(r_real))} of 2000 shuffles reach it")

naive = x0.std(ddof=1) / np.sqrt(n0)
blocks = x0.reshape(-1, ____).mean(axis=1)                                   # means of blocks of 50
honest = blocks.std(ddof=1) / np.sqrt(len(blocks))
print(f"mean {x0.mean():.3f}; naive SE {naive:.3f}; block SE {honest:.3f} ({honest / naive:.1f} times larger)")

**Check:** r₁ = 0.80, and none of the 2000 shuffles comes close (95% lie within ±0.04), so p < 0.0005. Naive standard error 0.037; blocks of 50 give 0.094, about 2.5 times larger. Heartbeats remember each other too: in Exercise 2, σ/√N will be far too small.

## <span style="color:#B5121B">1. Which frequencies, and when?</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 3 (indexing and slicing); Section 6 (functions); Section 8 (plots that say something). Each section takes 10–15 minutes.

These are Lecture 6's examples (slides 14, 16 and 20–22): reproduce the figures yourself.

**(a) Reading a spectrum, and how close two frequencies can be.** Lecture 6, slides 14 and 16. First the lecture's running example: amplitude 1 at 0.25 Hz and 0.5 at 1.1 Hz, plus noise, sampled every 0.1 s for 60 s. Read both amplitudes off the amplitude spectrum ($2|\mathrm{rfft}|/N$). Then two equal rhythms at 1.10 and 1.15 Hz: are they two peaks with 60 s of data? With 10 s? Does padding the 10-s record with zeros to 60 s (`np.fft.rfft(y, n=600)`) separate them?

In [ ]:
fs6 = 10.0                                                            # Lecture 6's sampling: every 0.1 s
t6 = np.arange(0, 60, 1 / fs6)
x6 = np.cos(2 * np.pi * 0.25 * t6) + 0.5 * np.cos(2 * np.pi * 1.1 * t6 + 1.0) + 0.4 * np.random.default_rng(6).standard_normal(len(t6))

def amplitude_spectrum(x, fs, n=None):
    """Amplitude of each frequency (Hz): 2|FFT|/N, after removing the mean. n > len(x) pads with zeros."""
    X = np.fft.rfft(x - x.mean(), n=n)
    return np.fft.rfftfreq(n or len(x), 1 / fs), 2 * np.abs(X) / len(x)

In [ ]:
f, amp = amplitude_spectrum(x6, fs6)
for f0 in (0.25, 1.1):
    k = np.argmin(np.abs(f - ____))                                   # the frequency closest to f0
    print(f"{f0} Hz: amplitude {amp[k]:.3f}")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
for k, (T, n) in enumerate([(60, None), (10, None), (10, 600)]):     # record length (s); padded length
    tt6 = np.arange(0, T, 1 / fs6)
    y = np.cos(2 * np.pi * 1.10 * tt6) + np.cos(2 * np.pi * 1.15 * tt6)
    f, amp = amplitude_spectrum(y, fs6, n=____)
    ax[k].plot(f, amp, ".-"); ax[k].set(xlim=(0.8, 1.6), xlabel="Frequency (Hz)", title=f"T = {T} s" + (", padded" if n else ""))
plt.tight_layout(); plt.show()

**Check:** amplitude 1.000 at 0.25 Hz and 0.495 at 1.1 Hz (true 1 and 0.5), on a noise level of about 0.03. With 60 s the 1.10 and 1.15 Hz rhythms are two peaks (resolution 1/T = 0.017 Hz); with 10 s (0.1 Hz) they are one. Padding to 60 s gives a finer grid but still **one** peak: resolution comes from the length of the data, not the number of points. To separate frequencies δf apart, record for T ≳ 1/δf.

**(b) The spectrum says which frequencies, not when.** Signal **A** contains 1.0 and 1.2 Hz together for 60 s; signal **B** is 1.0 Hz for 30 s, then 1.2 Hz. Compute the amplitude spectrum of each ($2|\mathrm{rfft}|/N$ against `rfftfreq`) and plot both. Could you tell them apart from their spectra?

In [ ]:
t1 = np.arange(0, 60, 1 / FS)
xA = 0.5 * np.sin(2 * np.pi * 1.0 * t1) + 0.5 * np.sin(2 * np.pi * 1.2 * t1)
xB = np.where(t1 < 30, np.sin(2 * np.pi * 1.0 * t1), np.sin(2 * np.pi * 1.2 * t1))

In [ ]:
for name, x in [("A", xA), ("B", xB)]:
    f, amp = amplitude_spectrum(____, FS)
    plt.plot(f, amp, label=name)
plt.xlim(0.6, 1.6); plt.xlabel("Frequency (Hz)"); plt.ylabel("Amplitude"); plt.legend(); plt.show()

**Check:** both spectra have peaks at 1.0 and 1.2 Hz with amplitude 0.5: the same spectrum for two very different signals. A spectrum averages over the whole record; it can't say *when* each frequency was there.

**(c) The short-time Fourier transform (STFT): the time–frequency trade-off.** Slide a window of length $L$ along the record and take the spectrum of the data inside it: a **spectrogram**. Below, a rhythm whose frequency swings, $f(t) = 1.1 + 0.2\sin(2\pi t/60\,\mathrm{s})$ Hz, plus noise, for 240 s. Compute spectrograms with $L$ = 3, 12 and 40 s. Lecture 6: the resolution is about $L$ in time and $1/L$ in frequency. Which window follows the swing best?

In [ ]:
rng1 = np.random.default_rng(3)
t2 = np.arange(0, 240, 1 / FS)
f_true = 1.1 + 0.2 * np.sin(2 * np.pi * t2 / 60)                      # Hz: the frequency at each moment
x_drift = np.sin(2 * np.pi * np.cumsum(f_true) / FS) + 0.3 * rng1.standard_normal(len(t2))

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

```python
n = int(L * FS)                                   # window length in samples
f, tt, S = signal.spectrogram(x, fs=FS, window="hann", nperseg=n, noverlap=n // 2)
plt.pcolormesh(tt, f, S, shading="auto")          # time across, frequency up
```
`S` has one column per window: `S[:, j]` is the spectrum of the window centred at `tt[j]`. The frequency with the most power in each window is `f[np.argmax(S, axis=0)]`.

</details>

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for k, L in enumerate([3, 12, 40]):                                   # window length (s)
    n = int(____ * FS)                                                # ... in samples
    f, tt, S = signal.spectrogram(x_drift, fs=FS, window="hann", nperseg=n, noverlap=n // 2)
    ax[k].pcolormesh(tt, f, S, shading="auto")
    ax[k].plot(t2, f_true, "w:", lw=1)                                # the true frequency
    track = f[np.argmax(S, axis=0)]                                   # the strongest frequency in each window
    err = np.sqrt(np.mean((track - (1.1 + 0.2 * np.sin(2 * np.pi * tt / 60)))**2))
    ax[k].set(ylim=(0.6, 1.6), xlabel="Time (s)", ylabel="Frequency (Hz)", title=f"L = {L} s: rms error {err:.3f} Hz")
plt.tight_layout(); plt.show()

**Check:** rms error about 0.09 Hz for L = 3 s (it follows the swing, but in steps of 0.33 Hz), 0.03 Hz for 12 s, 0.05 Hz for 40 s (sharp in frequency, but each window smears a large part of the 60-s swing). 12 s gives both.

> **Primer: averaging spectra.** A single spectrum of the whole record (a *periodogram*) has a surprising property: its scatter does not shrink as the record gets longer. A longer record gives finer frequency resolution, but each frequency is still estimated from one number, which scatters by about 100% of its own size. The STFT gives a way out: take the spectra of the windows and **average them over time**. With $K$ windows the scatter falls by about $\sqrt{K}$, and the resolution is about $1/L$: you trade resolution for smoothness. This averaged spectrum is the standard estimate of a spectrum from noisy data. In SciPy it is one function, `signal.welch` (named after Peter Welch, who proposed it in 1967); the one choice you must make, and justify, is the window length `nperseg`.

**(d) A broad peak is not necessarily noise.** Lecture 6, slide 22. Below is a second signal: a steady 1.1 Hz rhythm whose phase receives small random kicks. Average the spectrogram of each signal over time, using 40-s windows, and check that it is the same as `signal.welch`. From the averaged spectra alone, what would you say each signal contains? Then look at both spectrograms with 12-s windows: what is really there?

In [ ]:
rng3 = np.random.default_rng(4)
kicks = np.cumsum(np.sqrt(3.0 / FS) * rng3.standard_normal(len(t2)))            # a random walk of the phase (rad)
x_kick = np.sin(2 * np.pi * 1.1 * t2 + kicks) + 0.3 * rng3.standard_normal(len(t2))

In [ ]:
n = 160                                                              # 40-s windows
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for name, x, c in [("swinging frequency", x_drift, "#555656"), ("random phase kicks", x_kick, "#B5121B")]:
    f, tt, S = signal.spectrogram(x, fs=FS, window="hann", nperseg=n, noverlap=n // 2)
    P_avg = S.mean(axis=____)                                        # average over time: which axis is time?
    fw, Pw = signal.welch(x, fs=FS, nperseg=n)
    print(f"{name}: largest difference from welch {np.max(np.abs(P_avg - Pw)) / Pw.max():.1e} of the peak")
    ax[0].plot(f, P_avg, color=c, label=name)
for k, x in enumerate([x_drift, x_kick]):                            # now 12-s windows, to look in time
    f, tt, S = signal.spectrogram(x, fs=FS, window="hann", nperseg=48, noverlap=24)
    ax[k + 1].pcolormesh(tt, f, S, shading="auto"); ax[k + 1].set(ylim=(0.6, 1.6), xlabel="Time (s)")
ax[0].set(xlim=(0.6, 1.6), xlabel="Frequency (Hz)", ylabel="Averaged spectrum"); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Check:** the time-average of the spectrogram is identical to `signal.welch` (difference 0, or about 10⁻¹⁶ from rounding). Both averaged spectra spread over about 0.9–1.3 Hz. The swinging rhythm shows two peaks near 0.92 and 1.28 Hz, which look like two separate rhythms; the kicked one shows a single ragged hump. The 12-s spectrograms show what is really there: one rhythm swinging regularly every 60 s (it lingers at the ends of its swing, hence the two peaks), and one wandering at random.

**(e) Optional: aliasing.** Sample a 3.0 Hz sine at 4 samples per second for 20 s and find the peak of its spectrum. Where does it appear, and why? (Lecture 6: anything above the Nyquist frequency $f_s/2$ folds back.)

In [ ]:
# Optional

**(f) Optional: windows and leakage.** Lecture 6: a rhythm that doesn't fit a whole number of cycles into the record leaks power into every frequency. Take 60 s of a 1.05 Hz sine (exactly 63 cycles) and of a 1.0583 Hz sine (63.5 cycles), and plot their power spectra on a log axis, without and with a Hann window (`np.hanning(N)`, multiplied into the data before the FFT). What does the window do to the leakage, and what does it cost?

In [ ]:
# Optional

## <span style="color:#B5121B">2. From events to a regular series</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 3 (indexing and slicing); Section 4 (Boolean masks); Section 9 (just enough pandas, for the rolling median). Each section takes 10–15 minutes.

Many records are lists of **events**: heartbeats, neuron spikes, photon arrivals. The intervals between events vary, but they are not evenly spaced in time, so you can't take an FFT of them directly.

Below, an oscillator "beats" at about once a second, but its rate is modulated: a slow drift, plus a rhythm at **0.22 Hz** that you will try to recover. One beat has been missed by the detector.

**(a)** Compute the intervals between successive beats (`np.diff`), and plot them against the time of the beat that ends each one. Find the missed beat with last week's rule (more than 20% from the median of the 11 neighbouring intervals), and remove that interval.

In [ ]:
rng2 = np.random.default_rng(2)
dt_fine = 0.001
tf = np.arange(0, 900, dt_fine)
rate = 1.0 + 0.08 * np.sin(2 * np.pi * 0.22 * tf) + 0.05 * np.sin(2 * np.pi * 0.004 * tf)   # beats per second
phase = np.cumsum(rate) * dt_fine
idx = np.flatnonzero(np.diff(np.floor(phase)) > 0) + 1      # a beat each time the phase passes a whole number
beats = tf[idx] + 0.005 * rng2.standard_normal(len(idx))     # plus 5 ms of timing jitter
beats = np.delete(beats, 400)                        # the detector misses one beat
print(f"{len(beats)} beats in {beats[-1]:.0f} s")

In [ ]:
t_iv, iv = beats[1:], np.diff(____)                     # each interval, and the time of the beat that ends it
med = pd.Series(iv).rolling(11, center=True, min_periods=1).median().to_numpy()
bad = np.abs(iv - med) > ____ * med                      # last week's rule: more than 20% from the local median
print(bad.sum(), "flagged at t =", np.round(t_iv[bad], 1), "s")
plt.figure(figsize=(10, 3)); plt.plot(t_iv, iv, lw=0.6); plt.plot(t_iv[bad], iv[bad], "o", mfc="none")
plt.xlabel("Time (s)"); plt.ylabel("Interval (s)"); plt.show()
t_iv, iv = t_iv[~bad], iv[~bad]                          # keep the rest

**Check:** 902 beats; 1 interval flagged, 1.99 s long, at t = 398 s; 900 intervals remain.

**(b) Onto a regular grid.** Interpolate the kept intervals onto a grid of 4 samples per second, from the first to the last beat, with `np.interp` (linear) and with `CubicSpline`. How different are they (rms difference), compared with how much the intervals themselves vary?

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

```python
grid = np.arange(np.ceil(t_iv[0]), np.floor(t_iv[-1]), 1 / FS)
iv_lin = np.interp(grid, t_iv, iv)
iv_cub = CubicSpline(t_iv, iv)(grid)
```
The interpolated series is *the interval*, now known every 0.25 s instead of once per beat.

</details>

In [ ]:
grid = np.arange(np.ceil(t_iv[0]), np.floor(t_iv[-1]), 1 / FS)    # 4 samples per second
iv_lin = np.interp(____, t_iv, iv)
iv_cub = CubicSpline(t_iv, iv)(____)
print(f"rms difference {1000 * np.sqrt(np.mean((iv_cub - iv_lin)**2)):.1f} ms; the intervals vary by {1000 * iv.std():.0f} ms")

**Check:** the two interpolations differ by about 9 ms rms, against about 63 ms of real variation: either is fine.

**(c) The spectrum.** Take the averaged spectrum (`signal.welch`) of the interpolated intervals (subtract the mean; `nperseg=1024`, 256-s windows). Find the highest peak above 0.1 Hz. Did you recover the 0.22 Hz rhythm? Plot the spectrum with that frequency marked.

In [ ]:
f2, P2 = signal.welch(1000 * (iv_cub - iv_cub.mean()), fs=FS, nperseg=____)   # intervals in ms; 256 s segments
m = f2 > 0.1
print(f"peak at {f2[m][np.argmax(P2[m])]:.3f} Hz")
plt.semilogy(f2, P2); plt.axvline(0.22, ls=":"); plt.xlim(0, 0.6)
plt.xlabel("Frequency (Hz)"); plt.ylabel("PSD (ms$^2$/Hz)"); plt.show()

**Check:** the peak is at 0.219 Hz, within one resolution step (0.004 Hz) of 0.22 Hz.

**(d) Band power.** Integrate the spectrum from 0.15 to 0.30 Hz with **your own** trapezoid function (Week 3). This is the variance of the intervals that comes from that rhythm. Compare it with what you expect: a sine of amplitude $A$ has variance $A^2/2$, and here the intervals swing by about $A \approx 0.08 \times$ (mean interval)$^2 \times 1000$ ms.

In [ ]:
def trapezoid(y, x):
    """Your own trapezoid rule (Week 3): the sum of (width x average height) over the intervals."""
    return ____

band = (f2 >= 0.15) & (f2 <= 0.30)
A = 0.08 * iv.mean()**2 * 1000
print(f"band power {trapezoid(P2[band], f2[band]):.0f} ms^2; expected about {A**2 / 2:.0f} ms^2")

**Check:** about 2700 ms² from the spectrum, against about 3150 ms² expected. The small shortfall is real: each interval averages the rate over its own one-second length, which slightly smooths a 4.5-second rhythm.

**(e) Optional: the mistake to avoid.** Take the averaged spectrum of the intervals *without* interpolating, treating them as if they were evenly spaced at 4 Hz. Where does the peak appear now, and why is the frequency axis meaningless?

In [ ]:
# Optional

## <span style="color:#B5121B">3. Is the relationship real?</span>

> **Python help:** if any of the code below is unfamiliar, the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) practises it one idea at a time: Section 5 (loops that fill an array); Section 6 (functions); Section 7 (functions as inputs). Each section takes 10–15 minutes.

A "breathing" signal (an irregular rhythm near 0.25 Hz) and two beating oscillators: in **A** the beat rate follows the breathing; in **B** it follows a different, independent signal with exactly the same kind of rhythm. Both rates therefore wobble at about 0.25 Hz. Only one is driven by the breathing: we built it that way, so here we know. With real data you don't, and a cross-correlation can show that two signals vary together, with a lag, but not which drives which (Lecture 8, Part D shows a method that can).

> **Primer: cross-correlation.** Standardise both signals (subtract the mean, divide by the standard deviation). The cross-correlation at lag $\tau$ is the ordinary correlation coefficient between $x(t)$ and $y(t+\tau)$: slide one signal past the other and ask how alike they are at each shift. It runs from −1 to +1, and the lag of the largest |value| says which signal leads and by how much (here, a positive lag means the rate follows the breathing). Two signals with the same rhythm always correlate a little by chance, which is why (b) asks what chance gives.

**(a) Cross-correlation.** For each, put the rate (60 / interval, beats per minute) on the 4 Hz grid, standardise it and the breathing, and compute their correlation at lags from −5 s to +5 s. Report the largest |correlation| and its lag.

In [ ]:
rng3 = np.random.default_rng(3)
tb = np.arange(0, 900, 1 / FS)
sos = signal.butter(2, [0.18, 0.32], btype="band", fs=FS, output="sos")
def rhythm(r):                                   # an irregular rhythm near 0.25 Hz, standardised
    y = signal.sosfiltfilt(sos, r.standard_normal(len(tb))); return y / y.std()
breath, other = rhythm(rng3), rhythm(rng3)
sos_slow = signal.butter(2, [0.02, 0.15], btype="band", fs=FS, output="sos")
wander = signal.sosfiltfilt(sos_slow, rng3.standard_normal(len(tb))); wander /= wander.std()   # slow changes unrelated to breathing

def beats_from(driver, strength=0.06):
    """Beat times from a rate of 1 per second modulated by `driver` (on the tb grid)."""
    tfine = np.arange(0, 900, 0.001)
    r = 1.0 + strength * np.interp(tfine, tb, driver) + 0.06 * np.interp(tfine, tb, wander)
    ph = np.cumsum(r) * 0.001
    return tfine[np.flatnonzero(np.diff(np.floor(ph)) > 0) + 1]

beats_A, beats_B = beats_from(breath), beats_from(other)      # A: coupled to breathing; B: not

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

```python
def rate_on_grid(beats):
    t, iv = beats[1:], np.diff(beats)
    g = np.arange(np.ceil(t[0]), np.floor(t[-1]), 1 / FS)
    return g, 60 / CubicSpline(t, iv)(g)

def xcorr(x, y, maxlag):
    # correlation of x(t) with y(t + lag), lag in samples
    n = len(x); lags = np.arange(-maxlag, maxlag + 1)
    return lags, np.array([np.mean(x[max(0, -L):n - max(0, L)] * y[max(0, L):n - max(0, -L)]) for L in lags])
```
Standardise first: `(x - x.mean()) / x.std()`. Put the breathing on the same grid with `np.interp(g, tb, breath)`.

</details>

In [ ]:
def rate_on_grid(beats):
    """Beat rate (per minute) on a regular 4 Hz grid, from the beat times."""
    t, iv = beats[1:], np.diff(beats)
    g = np.arange(np.ceil(t[0]), np.floor(t[-1]), 1 / FS)
    return g, 60 / CubicSpline(t, iv)(g)

def std(x): return (x - x.mean()) / x.std()             # standardise: mean 0, standard deviation 1

def xcorr(x, y, maxlag=int(5 * FS)):
    """Correlation of x(t) with y(t + lag), for lags -maxlag..maxlag samples (positive: y follows x)."""
    n = len(x); lags = np.arange(-maxlag, maxlag + 1)
    return lags / FS, np.array([np.mean(x[max(0, -L):n - max(0, L)] * y[max(0, L):n - max(0, -L)]) for L in lags])

def strongest(x, y):
    """The largest |correlation| and its lag (s)."""
    lag, c = xcorr(x, y); k = np.argmax(np.abs(c)); return abs(c[k]), lag[k]

pairs = {}
for name, bt in [("A", beats_A), ("B", beats_B)]:
    g, hr = rate_on_grid(bt)
    x, y = std(np.interp(g, tb, ____)), std(hr)         # the breathing, on the same grid as the rate
    pairs[name] = (x, y)
    print(name, "largest |correlation| %.2f at a lag of %+.2f s" % strongest(x, y))

**Check:** A: about 0.67 at a lag of +0.5 s; B: about 0.07, at whatever lag chance picks.

**(b) A surrogate test.** Is B's largest correlation more than chance? Shift the breathing in time by a random amount of at least 30 s (`np.roll`, which wraps the end round to the start), recompute the largest |correlation|, and repeat 500 times. Where do A and B fall in this null distribution? What is each p-value?

<details><summary><b>Stuck? Open a hint</b> (try for 10 minutes first)</summary>

A circular shift keeps everything about each signal (its rhythm, its spectrum) and destroys only the timing between them:
```python
rng_s = np.random.default_rng(30)
null = [strongest(np.roll(x, rng_s.integers(120, len(x) - 120)), y)[0] for _ in range(500)]
p = (1 + np.sum(np.array(null) >= observed)) / (1 + len(null))
```

</details>

In [ ]:
rng_s = np.random.default_rng(30)
for name in ["A", "B"]:
    x, y = pairs[name]
    observed = strongest(x, y)[0]
    shift = lambda: rng_s.integers(int(30 * FS), len(x) - int(30 * FS))     # a random shift of at least 30 s
    null = np.array([strongest(np.roll(x, shift()), y)[0] for _ in range(____)])
    p = (1 + np.sum(null >= observed)) / (1 + len(null))
    print(f"{name}: observed {observed:.2f}; surrogate 95th percentile {np.percentile(null, 95):.2f}; p = {p:.3f}")

**Check:** A: p ≈ 0.002 (no surrogate comes close); B: p ≈ 0.5. Two rhythms at the same frequency always correlate a little (the surrogates' 95th percentile is about 0.12); the surrogates tell you how much is chance.

**(c) Optional: the wrong null.** Repeat (b) for B, but **shuffle** the breathing samples (`rng.permutation`) instead of shifting them. What p-value do you get now, and why is this test wrong for time series? (Lecture 5: shuffling destroys the autocorrelation as well as the timing.)

*Isn't shuffling what we did in Section 0?* Yes, and there it was right. A shuffle asks "is there any order at all?" (Lecture 5, slide 22). Here we already know each signal has a rhythm; the question is whether the *relationship* is more than two rhythms alone would give, so the surrogate must keep each signal's rhythm and break only their alignment: shift, don't shuffle. Lecture 8 returns to this on heartbeats.

In [ ]:
# Optional

## <span style="color:#B5121B">4. How big is a difference between groups?</span>

*(Short; do it now if there's time, or before you start Exercise 2.)* Band powers from 14 people in group X and 16 in group Y. Powers like these are skewed: a few people have much more than the rest.

**(a)** Plot both groups, every person shown. Give the difference of the means with a 95% bootstrap interval (resample each group separately, 10 000 times).

**(b)** Repeat on the **logarithms** of the powers: the difference of mean logs is a ratio, $e^{\Delta}$. Which summary would you report, and why?

In [ ]:
rng4 = np.random.default_rng(4)
group_x = np.round(np.exp(rng4.normal(np.log(1500), 0.8, 14)))      # ms^2
group_y = np.round(np.exp(rng4.normal(np.log(400), 0.8, 16)))
print("X:", np.sort(group_x).astype(int)); print("Y:", np.sort(group_y).astype(int))

In [ ]:
# TODO: strip plot; bootstrap interval for mean(X) - mean(Y); the same on logs, as a ratio

**Check:** difference of means about 1500 ms², interval roughly 700 to 2400; on logs, X has about 5 times the power of Y, interval about 2.7 to 9.2.

## <span style="color:#FD9029">Optional: the Lancaster Quay tide in frequency</span>

The Week 1–2 tide record covers about 8 days at 15-minute intervals. The two main tides are M2 (period 12.42 h) and S2 (12.00 h). Take the amplitude spectrum (frequency in cycles per day). Can you separate M2 and S2? How long a record would you need?

In [ ]:
tide = pd.read_csv(data_path("lancaster_quay_tidal.csv"), parse_dates=["time"])
days = (tide["time"] - tide["time"].iloc[0]).dt.total_seconds().to_numpy() / 86400
level = tide["level_m"].to_numpy()
print(f"{len(level)} readings over {days[-1]:.1f} days")

In [ ]:
# Optional

## Summary

- **Independence:** a shuffle keeps the values and destroys the order; if the real series differs from its shuffles, successive values are related, and σ/√N is too small.
- **Which frequencies, and when:** a spectrum says which, not when. The STFT looks in windows of length $L$: about $L$ in time, $1/L$ in frequency.
- **Averaged spectra** (the STFT averaged over time; `signal.welch`) trade resolution for smoothness. Say which window length you chose and why. A broad peak can be noise, or a rhythm that changes: look in time.
- **Events → series:** interpolate the intervals onto a regular grid before any spectrum; remove artefacts first.
- **Band power** = the area under the spectrum = the variance from that range of frequencies.
- **Relationships:** a correlation between two rhythmic signals is never zero; a time-shift surrogate tells you what chance gives. Shuffling is the wrong null for time series.
- **Differences between groups:** quote the size with a bootstrap interval; for skewed quantities, work with logs (ratios).

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>